# 02 — Quantization: Tuning the Delta Threshold

This notebook sweeps δ values and finds the **optimal threshold** —
the "elbow" where sparsity is high but Recall@10 hasn't collapsed.

Plots produced:
1. Sparsity % vs δ
2. Recall@10 vs δ
3. Memory (MB) vs δ
4. Combined tradeoff curve

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from tqdm.notebook import tqdm

from src.quantize import quantize, sparsity, delta_sweep
from src.baseline import FAISSBaseline
from src.index import TernaryIndex
from src.benchmark import sample_query_embeddings, compute_recall_at_k

In [ ]:
# Load float32 embeddings
embeddings = np.load('../embeddings/float32/embeddings.npy').astype(np.float32)
print(f"Embeddings: {embeddings.shape}  ({embeddings.nbytes / 1e6:.1f} MB)")

In [ ]:
# Build FAISS baseline (ground truth)
baseline = FAISSBaseline()
baseline.add(embeddings)

# Sample 200 queries for the sweep (faster)
queries, _ = sample_query_embeddings(embeddings, n=200)
print(f"Ground truth FAISS index ready. {len(queries)} query vectors.")

## Delta sweep

In [ ]:
DELTAS = np.linspace(0.01, 0.50, 40)
K = 10

# Ground truth top-k
true_indices, _ = baseline.batch_search(queries, k=K)

results = []
for d in tqdm(DELTAS, desc="Sweeping δ"):
    ternary = quantize(embeddings, float(d))
    idx = TernaryIndex(delta=float(d))
    idx.add_precomputed(ternary)
    pred_indices, _ = idx.batch_search(queries, k=K)
    
    results.append({
        "delta": float(d),
        "sparsity": float((ternary == 0).mean()),
        "memory_mb": ternary.nbytes / 1e6,
        "recall": compute_recall_at_k(true_indices, pred_indices, k=K),
    })

import pandas as pd
df = pd.DataFrame(results)
print(df.head(10))

## Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Delta Threshold Sweep', fontsize=14, fontweight='bold')

# 1. Sparsity vs delta
ax = axes[0, 0]
ax.plot(df['delta'], df['sparsity'] * 100, color='steelblue', linewidth=2)
ax.set_xlabel('δ (threshold)')
ax.set_ylabel('Sparsity (%)')
ax.set_title('Sparsity vs δ')
ax.grid(True, alpha=0.3)

# 2. Recall vs delta
ax = axes[0, 1]
ax.plot(df['delta'], df['recall'] * 100, color='coral', linewidth=2)
ax.axhline(88, color='gray', linestyle='--', label='88% target')
ax.set_xlabel('δ (threshold)')
ax.set_ylabel('Recall@10 (%)')
ax.set_title('Recall@10 vs δ')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Memory vs delta
ax = axes[1, 0]
ax.plot(df['delta'], df['memory_mb'], color='purple', linewidth=2)
ax.axhline(embeddings.nbytes / 1e6, color='gray', linestyle='--', label=f'Float32 ({embeddings.nbytes/1e6:.0f} MB)')
ax.axhline(20, color='red', linestyle=':', label='20 MB target')
ax.set_xlabel('δ (threshold)')
ax.set_ylabel('Memory (MB)')
ax.set_title('Memory vs δ')
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Recall vs Sparsity (tradeoff curve)
ax = axes[1, 1]
sc = ax.scatter(df['sparsity'] * 100, df['recall'] * 100,
                c=df['delta'], cmap='viridis', s=60, zorder=3)
plt.colorbar(sc, ax=ax, label='δ value')
ax.axhline(88, color='red', linestyle='--', alpha=0.7, label='88% recall target')
ax.set_xlabel('Sparsity (%)')
ax.set_ylabel('Recall@10 (%)')
ax.set_title('Recall vs Sparsity (tradeoff)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/delta_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/delta_sweep.png")

## Choose optimal delta

In [ ]:
# Find the largest δ where Recall@10 >= 88%
above_threshold = df[df['recall'] >= 0.88]

if len(above_threshold) > 0:
    optimal = above_threshold.iloc[-1]  # largest δ still meeting recall target
    print(f"Optimal δ = {optimal['delta']:.3f}")
    print(f"  Sparsity  : {optimal['sparsity']:.1%}")
    print(f"  Recall@10 : {optimal['recall']:.1%}")
    print(f"  Memory    : {optimal['memory_mb']:.1f} MB")
    print(f"  Compression: {embeddings.nbytes / 1e6 / optimal['memory_mb']:.1f}x")
    OPTIMAL_DELTA = float(optimal['delta'])
else:
    print("Warning: no δ achieves 88% recall. Using δ=0.05 as fallback.")
    OPTIMAL_DELTA = 0.05

print(f"\nUsing OPTIMAL_DELTA = {OPTIMAL_DELTA}")

In [ ]:
# Save the ternary index with optimal delta
from src.quantize import save_ternary

ternary = quantize(embeddings, OPTIMAL_DELTA)
save_ternary(ternary, OPTIMAL_DELTA)

print(f"\nFinal ternary index stats:")
print(f"  δ          : {OPTIMAL_DELTA}")
print(f"  Shape      : {ternary.shape}")
print(f"  Sparsity   : {float((ternary == 0).mean()):.1%}")
print(f"  Memory     : {ternary.nbytes / 1e6:.1f} MB")
print(f"  Float32 was: {embeddings.nbytes / 1e6:.1f} MB")

In [ ]:
# Save sweep results
Path('../results').mkdir(exist_ok=True)
df.to_csv('../results/delta_sweep.csv', index=False)
print("Saved: results/delta_sweep.csv")